# Reference facts from the laureate record for the people stage

Step 5 of the Econ forecast (`../README.md`): before the candidates per field are discussed, the record of the 99
laureates of 1969–2025 is examined for regularities that should inform the people stage (candidate pools weighted by
the Preseen field forecast, then a forecast of the laureate set). Each section ends with a short English paragraph
written to `prompt/<section>.md`, so that the facts can be attached to the people question as a context note.

**Inputs**

| File | Content |
|---|---|
| `../data/econ_prizes_laureates.csv` | every laureate 1969–2025: share, official motivation, affiliation, gender, birth and death dates, age at the award (crawl of 9 October 2026) |
| `../04_field_forecast/results/prize_works_fields14.csv` | the 64 prize works with their field (the 14 fields of Dolton and Tol) and the kind of contribution (step 2 consensus) |

**Sections:** 1 age at the award (distribution, is there an optimal age, change over time, by field and kind of
prize, co-laureate age gaps, the survival constraint).

**Data correction.** The Nobel Prize API dates the 2022 prize (Bernanke, Diamond, Dybvig) 2011-10-10, so the
`age_at_award` column of the crawl is eleven years too low for these three laureates (57, 57, 56 instead of 68, 68,
67). Ages here are recomputed from the birth date to the announcement in the prize year; the crawled file stays as
fetched. The step-2 notebook's age section used the API ages and is corrected the same way.

Figures and tables are written to `results/`; the prompt paragraphs to `prompt/`.

In [ ]:
from fractions import Fraction
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from sklearn.mixture import GaussianMixture

ECON = Path('/project/jevans/Dawoon/Nobel Prize/Econ')
HERE = ECON / '05_laureates'
OUT, PROMPT = HERE / 'results', HERE / 'prompt'
for d in (OUT, PROMPT):
    d.mkdir(parents=True, exist_ok=True)
FORECAST_YEAR = 2026
ANNOUNCE = pd.Timestamp('2026-10-12')           # the 2026 announcement (earliest date)
CUTOFF = '2026-10-09'
rng = np.random.default_rng(2026)
pd.set_option('display.max_colwidth', 110, 'display.width', 220, 'display.max_rows', 120)

# Chart style: the dataviz reference palette (light), hairline grid, thin marks, text in ink tokens (as in 02_fields)
INK, INK2, MUTED, GRID, BASE, SURFACE = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7', '#fcfcfb'
BLUE, ORANGE, AQUA, YELLOW = '#2a78d6', '#eb6834', '#1baf7a', '#eda100'      # categorical slots 1-4, fixed order
BLUE_LIGHT, GRAY = '#b7d3f6', '#c3c2b7'                                     # sequential blue 150; de-emphasis
mpl.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE, 'figure.dpi': 110,
    'axes.edgecolor': BASE, 'axes.linewidth': 1, 'axes.labelcolor': INK2, 'axes.titlecolor': INK,
    'axes.titlesize': 12, 'axes.titleweight': 'semibold', 'axes.titlelocation': 'left', 'axes.labelsize': 10,
    'xtick.color': BASE, 'ytick.color': BASE, 'xtick.labelcolor': INK2, 'ytick.labelcolor': INK2,
    'xtick.labelsize': 9, 'ytick.labelsize': 9, 'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8,
    'axes.axisbelow': True, 'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False,
    'legend.fontsize': 9, 'font.family': 'sans-serif', 'svg.fonttype': 'none', 'lines.linewidth': 2,
    'lines.solid_capstyle': 'round'})


def save(fig, name):
    fig.savefig(OUT / f'{name}.svg', bbox_inches='tight')
    fig.savefig(OUT / f'{name}.png', bbox_inches='tight', dpi=160)


# ---- laureates
L = pd.read_csv(ECON / 'data/econ_prizes_laureates.csv')
DEATH_ADDED = {'Oliver E. Williamson': '2020-05-21'}     # missing in the API (crawl of 9 Oct 2026); UC Berkeley obituary
for name, d in DEATH_ADDED.items():
    L.loc[L['laureate'].eq(name) & L['death_date'].isna(), 'death_date'] = d
L['work_id'] = L['year'].astype(str) + '-' + L['group'].astype(str)
L['share'] = L['portion'].map(lambda s: float(Fraction(s)))
L['alive'] = L['death_date'].isna()
L['woman'] = L['gender'].eq('female')
# exact age at the award: birth date (four laureates have a birth year only: mid-year, flagged approximate) to the
# announcement date; three API award dates fall in another year than the prize and are replaced by 10 October
born = pd.to_datetime(L['birth_date'].where(L['birth_date_precision'].eq('day'), L['birth_date'].str[:4] + '-07-01'))
awarded = pd.to_datetime(L['date_awarded'])
odd = awarded.dt.year != L['year']
awarded = awarded.where(~odd, pd.to_datetime(L['year'].astype(str) + '-10-10'))
L['age'] = (awarded - born).dt.days / 365.2425
L['age_approx'] = ~L['birth_date_precision'].eq('day')
# DATA ERROR in the Nobel Prize API (crawl of 9 Oct 2026): the 2022 prize is dated 2011-10-10, so the API's integer
# age_at_award of Bernanke, Diamond and Dybvig is eleven years too low (57, 57, 56). The ages here use the prize year.
print('award dates in another year than the prize (API error; the age is recomputed from 10 October of the prize year):')
display(L.loc[odd, ['year', 'laureate', 'birth_date', 'date_awarded', 'age_at_award']].assign(age_corrected=L.loc[odd, 'age'].round(1)))
ok = ~odd
assert (L.loc[ok, 'age'].round().sub(L.loc[ok, 'age_at_award']).abs() <= 1).all(), 'exact ages must agree with the API integer ages'
died = pd.to_datetime(L['death_date'])
L['age_at_death'] = (died - born).dt.days / 365.2425
L['years_after_prize'] = (died.fillna(ANNOUNCE) - awarded).dt.days / 365.2425
L['age_now'] = ((ANNOUNCE - born).dt.days / 365.2425).where(L['alive'])

# ---- works: field (14 fields of Dolton and Tol, step 4a) and kind of contribution (step 2)
W = pd.read_csv(ECON / '04_field_forecast/results/prize_works_fields14.csv')
L = L.merge(W[['work_id', 'field14', 'contribution']], on='work_id', how='left')
L['n_prize'] = L.groupby('year')['laureate'].transform('size')          # laureates sharing the prize year
L['n_work'] = L.groupby('work_id')['laureate'].transform('size')        # laureates sharing the work
SHORT = {'Econometrics': 'Econometrics', 'Equilibrium, Welfare': 'Equilibrium', 'Games, Market Structure': 'Games',
         'Behavioural, Experimental': 'Behavioural', 'Labour': 'Labour', 'Production, Industrial Organization': 'Production/IO',
         'Public, Law, Political Economy': 'Public', 'Information': 'Information', 'Macro': 'Macro', 'Trade': 'Trade',
         'Finance': 'Finance', 'Development, Economic History': 'Development/EH', 'Growth': 'Growth', 'Resources, Environment': 'Resources'}
L['field'] = L['field14'].map(SHORT)


def period(year):
    if year < 1980:
        return '1969–79'
    if year >= 2020:
        return '2020–25'
    return f'{year // 10 * 10}s'


PERIODS = ['1969–79', '1980s', '1990s', '2000s', '2010s', '2020–25']
L['period'] = pd.Categorical(L['year'].map(period), PERIODS, ordered=True)
assert len(L) == 99 and L['field14'].notna().all()
print(f"{len(L)} laureates, {L['work_id'].nunique()} works, {L['year'].nunique()} prizes; ages exact for {(~L.age_approx).sum()}, "
      f"approximate (birth year only) for {L.age_approx.sum()}")

## 1. Age at the award

**Definitions.** Age = years from the birth date to the announcement date of the prize. "Optimal age" can mean two
different things: (a) the age at which laureates were most often awarded (the mode of the laureate age distribution),
which this record measures directly; (b) the age at which a *candidate* is most likely to be awarded, which needs a
pool of candidates who did not win: Dolton and Tol (2026) estimate (b) on their constructed pool and find a quadratic
in age peaking at about 71 years (71.4 for 1969–1997, 70.8 for 1998–2025) [P]. Without their pool, the closest
measure here is the award rate among *eventual* laureates: of the laureates not yet awarded at age $a$, the share
awarded at that age. It rises mechanically late in life as few remain, so only the ages at which at least ten
laureates were still unawarded are read.

### 1.1 The distribution

In [ ]:
a = L['age']
q = a.quantile([0.10, 0.25, 0.50, 0.75, 0.90])
summary = pd.Series({'laureates': len(a), 'mean': a.mean(), 'standard deviation': a.std(), 'median': q[0.5],
                     'p10': q[0.1], 'p25': q[0.25], 'p75': q[0.75], 'p90': q[0.9], 'youngest': a.min(), 'oldest': a.max(),
                     'skewness': stats.skew(a)}).round(1)
display(summary.to_frame('age at the award').T)
BANDS = pd.cut(a, [0, 50, 55, 60, 65, 70, 75, 80, 101], right=False,
               labels=['under 50', '50-54', '55-59', '60-64', '65-69', '70-74', '75-79', '80 and over'])
band_tab = BANDS.value_counts().sort_index().to_frame('laureates')
band_tab['share'] = (band_tab['laureates'] / len(a)).map('{:.0%}'.format)
display(band_tab.T)
cols = ['year', 'laureate', 'age', 'field', 'n_prize']
display(pd.concat([L.nsmallest(5, 'age')[cols].assign(rank='youngest'), L.nlargest(5, 'age')[cols].assign(rank='oldest')])
          .set_index('rank').round(1))

fig, ax = plt.subplots(figsize=(8, 3.8))
bins = np.arange(45, 96, 5)
ax.hist(a, bins=bins, color=BLUE, edgecolor=SURFACE, linewidth=2, label='laureates (5-year bins)')
grid = np.linspace(45, 95, 400)
kde = stats.gaussian_kde(a)
ax.plot(grid, kde(grid) * len(a) * 5, color=INK2, linewidth=2, label='density (same scale)')
ax.axvline(q[0.5], color=INK, linestyle='--', linewidth=1.2)
ax.text(q[0.5] + 0.4, ax.get_ylim()[1] * 0.95, f'median {q[0.5]:.0f}', color=INK, fontsize=9, va='top')
ax.axvspan(70, 71.5, color=ORANGE, alpha=0.25, linewidth=0)
ax.text(71.8, ax.get_ylim()[1] * 0.78, 'Dolton–Tol peak 70–71\n(candidate-pool model) [P]', color=INK2, fontsize=8.5, va='top')
ax.set_xlabel('age at the award (years)')
ax.set_ylabel('laureates')
ax.grid(axis='x', visible=False)
ax.set_title('Age of the 99 laureates at the award, 1969–2025')
ax.legend(loc='upper left')
save(fig, 'age_distribution')
plt.show()

### 1.2 Is there an optimal age? Mode, unimodality and the award rate among eventual laureates

In [ ]:
# (a) the mode of the laureate age distribution, with a bootstrap interval
grid = np.arange(40, 95.01, 0.1)
mode = grid[kde(grid).argmax()]
B = 2000
boot_mode = np.array([grid[stats.gaussian_kde(rng.choice(a.to_numpy(), len(a), replace=True))(grid).argmax()] for _ in range(B)])
boot_median = np.median(rng.choice(a.to_numpy(), (B, len(a)), replace=True), axis=1)
ci = lambda v: tuple(np.percentile(v, [2.5, 97.5]))
print(f'mode of the density {mode:.1f} (bootstrap 95 % {ci(boot_mode)[0]:.1f}–{ci(boot_mode)[1]:.1f}); '
      f'median {a.median():.1f} ({ci(boot_median)[0]:.1f}–{ci(boot_median)[1]:.1f}); mean {a.mean():.1f}')

# (b) one peak or several? Gaussian mixtures with 1-3 components (BIC), skewness, normality
X = a.to_numpy().reshape(-1, 1)
gmm = {k: GaussianMixture(k, n_init=5, random_state=2026).fit(X) for k in (1, 2, 3)}
bic = pd.DataFrame({'components': list(gmm), 'BIC': [g.bic(X) for g in gmm.values()],
                    'means': [', '.join(f'{m:.0f}' for m in sorted(g.means_.ravel())) for g in gmm.values()],
                    'weights': [', '.join(f'{w:.2f}' for w in g.weights_[np.argsort(g.means_.ravel())]) for g in gmm.values()]}).set_index('components')
display(bic.round(1))
shapiro = stats.shapiro(a)
print(f'best by BIC: {bic.BIC.idxmin()} component(s); skewness {stats.skew(a):+.2f}; Shapiro–Wilk p = {shapiro.pvalue:.2f} '
      f'(normal shape not rejected)' if shapiro.pvalue > 0.05 else f'best by BIC: {bic.BIC.idxmin()}; skewness {stats.skew(a):+.2f}; Shapiro p = {shapiro.pvalue:.3f}')

# (c) award rate among eventual laureates: of those not yet awarded at age x, the share awarded at x
ages = np.arange(40, 96)
age_floor = np.floor(a).astype(int)
events = age_floor.value_counts().reindex(ages, fill_value=0)
at_risk = pd.Series({x: int((age_floor >= x).sum()) for x in ages})
rate = events / at_risk
smooth = rate.rolling(5, center=True, min_periods=3).mean()
ok = at_risk >= 10
peak_age = int(smooth[ok].idxmax())
HZ = pd.DataFrame({'awarded at this age': events, 'not yet awarded (at risk)': at_risk, 'rate': rate.round(3), 'rate, 5-year smoothed': smooth.round(3)})
HZ.index.name = 'age'
display(HZ.loc[50:85].T)
print(f'the smoothed award rate among eventual laureates first exceeds 5 % at age {int(smooth[smooth > 0.05].index.min())}, '
      f'10 % at {int(smooth[smooth > 0.10].index.min())}, and peaks at {peak_age} among ages with at least ten laureates still unawarded '
      f'({smooth[peak_age]:.0%} per year); Dolton and Tol: peak at 70–71 in the candidate pool [P]')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True, gridspec_kw={'height_ratios': [3, 1.3], 'hspace': 0.12})
ax1.plot(rate.index[ok], rate[ok], color=BLUE_LIGHT, linewidth=1.5, label='yearly rate')
ax1.plot(smooth.index[ok], smooth[ok], color=BLUE, linewidth=2.2, label='5-year smoothed')
ax1.axvspan(70, 71.5, color=ORANGE, alpha=0.25, linewidth=0)
ax1.text(71.8, 0.95 * smooth[ok].max(), 'Dolton–Tol peak [P]', color=INK2, fontsize=8.5, va='top')
ax1.axvline(peak_age, color=INK, linestyle='--', linewidth=1.2)
ax1.text(peak_age + 0.4, 0.6 * smooth[ok].max(), f'peak {peak_age}', color=INK, fontsize=9)
ax1.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1))
ax1.set_ylabel('share of the not-yet-awarded\nawarded at this age')
ax1.grid(axis='x', visible=False)
ax1.set_title('Award rate among eventual laureates, ages with at least ten laureates still unawarded')
ax1.legend(loc='upper left')
ax2.bar(at_risk.index[ok], at_risk[ok], color=GRAY, edgecolor=SURFACE, linewidth=1, width=1)
ax2.set_ylabel('still unawarded')
ax2.set_xlabel('age (years)')
ax2.grid(axis='x', visible=False)
save(fig, 'age_award_rate')
plt.show()

### 1.3 Over time

In [ ]:
per = L.groupby('period', observed=True)['age'].agg(laureates='size', median='median', p25=lambda s: s.quantile(.25),
                                                      p75=lambda s: s.quantile(.75), youngest='min', oldest='max').round(1)
display(per.T)
fit = stats.linregress(L['year'], L['age'])
rho = stats.spearmanr(L['year'], L['age'])
print(f'age on award year: {fit.slope * 10:+.2f} years per decade (p = {fit.pvalue:.2f}); Spearman rho = {rho.statistic:+.2f} (p = {rho.pvalue:.2f})')
age_early, age_late = L[L.year < 1998]['age'], L[L.year >= 1998]['age']
mw = stats.mannwhitneyu(age_early, age_late)
print(f'1969–1997: median {age_early.median():.1f} (n = {len(age_early)}); 1998–2025: median {age_late.median():.1f} (n = {len(age_late)}); '
      f'Mann–Whitney p = {mw.pvalue:.2f}; spread (IQR) {stats.iqr(age_early):.1f} vs {stats.iqr(age_late):.1f}')

S = L.sort_values(['year', 'age']).reset_index(drop=True)
roll = S['age'].rolling(15, center=True, min_periods=8)
fig, ax = plt.subplots(figsize=(8.5, 3.8))
ax.fill_between(S['year'], roll.quantile(0.25), roll.quantile(0.75), color=BLUE_LIGHT, alpha=0.6, linewidth=0, label='rolling interquartile range (15 laureates)')
ax.scatter(S['year'], S['age'], s=22, color=INK2, edgecolor=SURFACE, linewidth=1, zorder=3, label='laureate')
ax.plot(S['year'], roll.median(), color=BLUE, linewidth=2.2, zorder=4, label='rolling median')
ax.set_xlim(1967, 2027)
ax.set_xlabel('award year')
ax.set_ylabel('age at the award')
ax.grid(axis='x', visible=False)
ax.set_title('Age at the award over time: no trend')
ax.legend(loc='lower left', ncol=3)
save(fig, 'age_over_time')
plt.show()

### 1.4 By field, kind of contribution and size of the prize

In [ ]:
def agg(col, min_n=1):
    t = L.groupby(col, observed=True)['age'].agg(laureates='size', median='median', p25=lambda s: s.quantile(.25),
                                                 p75=lambda s: s.quantile(.75), youngest='min', oldest='max').round(1)
    return t[t.laureates >= min_n].sort_values('median')


by_field = agg('field')
display(by_field)
big = [f for f, n in L['field'].value_counts().items() if n >= 3]
kw = stats.kruskal(*[L.loc[L.field == f, 'age'] for f in big])
print(f'fields with at least three laureates ({len(big)}): Kruskal–Wallis p = {kw.pvalue:.2f}')
display(agg('contribution'))
kw2 = stats.kruskal(*[g['age'] for _, g in L.groupby('contribution')])
print(f'kind of contribution: Kruskal–Wallis p = {kw2.pvalue:.2f}')
display(agg('n_prize').rename_axis('laureates sharing the prize'))
kw3 = stats.kruskal(*[g['age'] for _, g in L.groupby('n_prize')])
print(f'laureates per prize: Kruskal–Wallis p = {kw3.pvalue:.3f}')
display(L[L.woman][['year', 'laureate', 'age', 'field', 'n_prize']].round(1).set_index('year'))

fig, ax = plt.subplots(figsize=(8, 4.6))
order = by_field.index.tolist()
ypos = {f: i for i, f in enumerate(order)}
jit = rng.uniform(-0.18, 0.18, len(L))
ax.scatter(L['age'], L['field'].map(ypos) + jit, s=18, color=GRAY, edgecolor=SURFACE, linewidth=0.8, zorder=2, label='laureate')
ax.scatter(by_field['median'], range(len(order)), s=70, color=BLUE, edgecolor=SURFACE, linewidth=1.5, zorder=3, label='median')
for i, f in enumerate(order):
    ax.text(97.5, i, f"n = {by_field.loc[f, 'laureates']}", va='center', fontsize=8.5, color=MUTED)
ax.axvline(a.median(), color=INK, linestyle='--', linewidth=1, zorder=1)
ax.set_yticks(range(len(order)), order)
ax.set_xlim(44, 101)
ax.set_xlabel('age at the award')
ax.grid(axis='y', visible=False)
ax.set_title('Age at the award by field (14 fields of Dolton and Tol), dashed line = overall median')
ax.legend(loc='lower right')
save(fig, 'age_by_field')
plt.show()

### 1.5 Age gaps between co-laureates

In [ ]:
shared = L[L.n_work >= 2].groupby('work_id').agg(year=('year', 'first'), laureates=('laureate', lambda s: ', '.join(s)),
                                                  n=('laureate', 'size'), youngest=('age', 'min'), oldest=('age', 'max'),
                                                  field=('field', 'first'))
shared['gap'] = shared['oldest'] - shared['youngest']
print(f"{len(shared)} works shared by two or three laureates: median age gap {shared.gap.median():.1f} years, "
      f"interquartile {shared.gap.quantile(.25):.1f}–{shared.gap.quantile(.75):.1f}; gap of 10+ years in {(shared.gap >= 10).sum()}, "
      f"15+ in {(shared.gap >= 15).sum()}, 20+ in {(shared.gap >= 20).sum()} of {len(shared)}")
display(shared.sort_values('gap', ascending=False).head(8).round(1))
same_gen = shared[shared.gap < 5]
print(f'co-laureates within five years of age: {len(same_gen)} of {len(shared)} works')
# divided prizes: the two works of one year
div = L[L.groupby('year')['group'].transform('nunique') > 1].groupby(['year', 'group'])['age'].median().unstack()
div['gap between the two works (medians)'] = (div[1] - div[2]).abs().round(1)
display(div.round(1))

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(shared['gap'], bins=np.arange(0, 41, 5), color=BLUE, edgecolor=SURFACE, linewidth=2)
ax.axvline(shared.gap.median(), color=INK, linestyle='--', linewidth=1.2)
ax.text(shared.gap.median() + 0.5, ax.get_ylim()[1] * 0.92, f'median {shared.gap.median():.0f}', color=INK, fontsize=9)
ax.set_xlabel('age gap between the oldest and the youngest co-laureate of a work (years)')
ax.set_ylabel('shared works')
ax.grid(axis='x', visible=False)
ax.set_title('Co-laureates are usually of one generation')
save(fig, 'colaureate_age_gap')
plt.show()

### 1.6 The survival constraint: the prize is not awarded posthumously

In [ ]:
dead, liv = L[~L.alive], L[L.alive]
print(f'deceased laureates: {len(dead)}; age at death median {dead.age_at_death.median():.1f} (range {dead.age_at_death.min():.0f}–{dead.age_at_death.max():.0f}); '
      f'years lived after the prize: median {dead.years_after_prize.median():.1f}, within 3 years {int((dead.years_after_prize < 3).sum())}, '
      f'within 5 years {int((dead.years_after_prize < 5).sum())}, within 10 years {int((dead.years_after_prize < 10).sum())}')
old = L[L.age >= 80].sort_values('age', ascending=False)
print(f'awarded at 80 or older: {len(old)} of 99')
display(old[['year', 'laureate', 'age', 'field', 'death_date', 'years_after_prize']].round(1).set_index('year'))
late75 = L[L.age >= 75]
print(f'awarded at 75 or older: {len(late75)} of 99; of them {int((~late75.alive).sum())} have died, after a median {late75.loc[~late75.alive, "years_after_prize"].median():.1f} years')
print(f'living laureates: {len(liv)}; current age median {liv.age_now.median():.1f}, 80 or older {int((liv.age_now >= 80).sum())}, 90 or older {int((liv.age_now >= 90).sum())}')
display(pd.cut(liv.age_now, [40, 60, 70, 80, 90, 101], right=False).value_counts().sort_index().rename('living laureates by current age').to_frame().T)

### 1.7 Summary and the prompt paragraph

In [ ]:
n80, n75, u55 = int((a >= 80).sum()), int((a >= 75).sum()), int((a < 55).sum())
rows = {
    'laureates': len(a),
    'median age (bootstrap 95 %)': f'{a.median():.1f} ({ci(boot_median)[0]:.1f}–{ci(boot_median)[1]:.1f})',
    'mean / standard deviation': f'{a.mean():.1f} / {a.std():.1f}',
    'middle 80 % (p10–p90)': f'{q[0.1]:.0f}–{q[0.9]:.0f}',
    'mode of the density (bootstrap 95 %)': f'{mode:.0f} ({ci(boot_mode)[0]:.0f}–{ci(boot_mode)[1]:.0f})',
    'components preferred by BIC': int(bic.BIC.idxmin()),
    'award rate among eventual laureates: first > 5 % / > 10 % / peak': f'{int(smooth[smooth > 0.05].index.min())} / {int(smooth[smooth > 0.10].index.min())} / {peak_age}',
    'Dolton–Tol candidate-pool peak [P]': '70–71 (71.4 for 1969–97, 70.8 for 1998–2025)',
    'trend per decade (p)': f'{fit.slope * 10:+.2f} ({fit.pvalue:.2f})',
    'under 55 / 75 and over / 80 and over': f'{u55} / {n75} / {n80}',
    'field medians: youngest, oldest (Kruskal–Wallis p)': f"{by_field.index[0]} {by_field['median'].iloc[0]:.0f}, {by_field.index[-1]} {by_field['median'].iloc[-1]:.0f} ({kw.pvalue:.2f})",
    'median by laureates per prize 1 / 2 / 3 (p)': ' / '.join(f"{agg('n_prize').loc[k, 'median']:.0f}" for k in (1, 2, 3)) + f' ({kw3.pvalue:.3f})',
    'co-laureate age gap: median, 15+ years': f'{shared.gap.median():.0f}, {(shared.gap >= 15).sum()} of {len(shared)} works',
    'deceased: age at death median, years after the prize median, died within 5 years': f'{dead.age_at_death.median():.1f}, {dead.years_after_prize.median():.1f}, {int((dead.years_after_prize < 5).sum())} of {len(dead)}',
}
SUM = pd.Series(rows).to_frame('value')
display(SUM)
SUM.to_csv(OUT / 'age_at_award_summary.csv')
L[['year', 'work_id', 'laureate', 'age', 'age_approx', 'field14', 'contribution', 'n_prize', 'n_work', 'alive', 'age_at_death', 'years_after_prize', 'age_now']].round(2).to_csv(OUT / 'laureate_ages.csv', index=False)
HZ.to_csv(OUT / 'age_award_rate.csv')

fields_txt = '; '.join(f"{f} {r['median']:.0f} (n = {r['laureates']})" for f, r in by_field.iterrows())
periods_txt = ', '.join(f"{p} {r['median']:.0f}" for p, r in per.iterrows())
old_txt = '; '.join(f"{r.laureate} {r.year} at {r.age:.0f}" for r in old.itertuples())
gaps_txt = '; '.join(f"{r.year} {r.laureates} ({r.gap:.0f} years)" for r in shared.sort_values('gap', ascending=False).head(3).itertuples())
text = f'''# Age at the award: the record of the 99 laureates, 1969-2025 [S, P]

Information cutoff: {CUTOFF}. Age = years from the birth date to the announcement of the prize (four laureates have a
birth year only; their age is approximate to half a year; the Nobel Prize API misdates the 2022 award as 2011, so the
ages of Bernanke, Diamond and Dybvig are taken from the prize year: 68, 68 and 67, not the API's 57, 57 and 56).

- Distribution: median {a.median():.0f} (bootstrap 95 % {ci(boot_median)[0]:.0f}-{ci(boot_median)[1]:.0f}), mean {a.mean():.1f}, standard deviation {a.std():.1f}; the middle
  80 % of laureates were {q[0.1]:.0f}-{q[0.9]:.0f} years old. Youngest {L.nsmallest(1, 'age').iloc[0].laureate} ({L.nsmallest(1, 'age').iloc[0].year}, {a.min():.0f}), oldest
  {L.nlargest(1, 'age').iloc[0].laureate} ({L.nlargest(1, 'age').iloc[0].year}, {a.max():.0f}). Under 55: {u55} of 99; 75 and over: {n75}; 80 and over: {n80} ({old_txt}).
- One peak, not several: a Gaussian mixture prefers {int(bic.BIC.idxmin())} component(s) by BIC; skewness {stats.skew(a):+.2f}; the mode of the
  density is about {mode:.0f} (bootstrap 95 % {ci(boot_mode)[0]:.0f}-{ci(boot_mode)[1]:.0f}). Among eventual laureates, the share of the not-yet-awarded who were
  awarded in a given year of age stays under 5 % before {int(smooth[smooth > 0.05].index.min())}, passes 10 % at {int(smooth[smooth > 0.10].index.min())} and peaks around
  {peak_age} (ages at which at least ten laureates were still unawarded). Dolton and Tol's candidate-pool logit, which also
  counts the people who never won, puts the probability of winning at its maximum at about 70-71 years of age [P].
  Read together: awards cluster in the sixties, the chance per year is highest around 70, and awards after 75 are
  uncommon but not rare.
- No trend over time: {fit.slope * 10:+.2f} years per decade (p = {fit.pvalue:.2f}); period medians {periods_txt}; 1969-1997 median
  {age_early.median():.0f} against 1998-2025 median {age_late.median():.0f} (p = {mw.pvalue:.2f}).
- By field (median age, laureates): {fields_txt}. The differences are
  not significant across fields with at least three laureates (Kruskal-Wallis p = {kw.pvalue:.2f}). By laureates per
  prize: solo {agg('n_prize').loc[1, 'median']:.0f}, two {agg('n_prize').loc[2, 'median']:.0f}, three {agg('n_prize').loc[3, 'median']:.0f} (p = {kw3.pvalue:.3f}): three-way prizes go to younger people.
- Co-laureates are usually of one generation: in the {len(shared)} works shared by two or three laureates the age gap between
  the oldest and the youngest has a median of {shared.gap.median():.0f} years; {int((shared.gap >= 15).sum())} works have a gap of 15 years or more
  ({gaps_txt}).
- The prize is not awarded posthumously: the {len(dead)} deceased laureates died at a median age of {dead.age_at_death.median():.0f}, a median
  {dead.years_after_prize.median():.0f} years after the prize; {int((dead.years_after_prize < 5).sum())} died within five years of it. Of the {n75} laureates awarded at 75 or
  older, {int((~late75.alive).sum())} have died, after a median {late75.loc[~late75.alive, "years_after_prize"].median():.0f} years. {len(liv)} laureates are alive (median age {liv.age_now.median():.0f}; {int((liv.age_now >= 80).sum())} are 80 or older).

Use for the people stage: a candidate's age in 2026 matters mostly at the extremes. Under 55 is rare ({u55} of 99) and
80 and over is rare ({n80} of 99) but has happened for long-recognized theoretical work; between 58 and 77 age alone
separates candidates little. Within one laureate set, expect co-laureates of one generation more often than a
mentor-student pair.
'''
(PROMPT / '01_age_at_award.md').write_text(text)
print(text)
print('written:', PROMPT / '01_age_at_award.md', 'and', ', '.join(sorted(p.name for p in OUT.iterdir())))